# 02 - Git and GitHub for AI Deployment

This chapter teaches Git and GitHub from scratch with deployment context.


## Section 1: Git Fundamentals

### Definition
Git is distributed version control system that tracks file changes over time.

### Theory
Every commit stores snapshot of project state and metadata (author, timestamp, message).

### Motivation
Deployment platforms pull exact commit state, not loose local files.

### Real-World Examples
- Roll back broken release to previous commit.
- Compare two model-serving implementations.

### Visual Explanation
![Git Workflow](../outputs/figures/git-workflow.png)

### Best Practices
- Use meaningful commit messages.
- Commit frequently in small logical units.

### Common Mistakes
- One huge commit after days of work.
- Ambiguous messages like `update`.


In [ ]:
from pathlib import Path
import subprocess
import tempfile

demo_dir = Path(tempfile.mkdtemp(prefix="git_demo_"))
print("Demo repo:", demo_dir)

commands = [
    ["git", "init"],
    ["git", "config", "user.name", "demo-user"],
    ["git", "config", "user.email", "demo@example.com"],
]
for cmd in commands:
    out = subprocess.run(cmd, cwd=demo_dir, capture_output=True, text=True, check=False)
    print("$", " ".join(cmd))
    print((out.stdout or out.stderr).strip())

(demo_dir / "README.md").write_text("# Demo Repo\n")
subprocess.run(["git", "add", "README.md"], cwd=demo_dir, check=False)
subprocess.run(["git", "commit", "-m", "feat: add README"], cwd=demo_dir, check=False)

log = subprocess.run(["git", "log", "--oneline", "-n", "1"], cwd=demo_dir, capture_output=True, text=True, check=False)
print("Latest commit:", log.stdout.strip())


## Section 2: Core Git Terms

### Definition and Motivation
- **Repository:** project folder tracked by Git.
- **Commit:** saved checkpoint.
- **Branch:** independent line of development.
- **Push:** upload commits to remote.
- **Pull:** sync remote changes locally.
- **Main branch:** default integration branch for releases.

### Theory
Branching allows feature work without destabilizing deployment branch.

### Real-World Example
Build feature in `feature/deploy-monitoring`, then merge to `main` only after tests pass.

### Code Explanation
Next cell prints branch state from current project, if available.

### Best Practices
- Protect `main` with CI checks.
- Keep branch names task-scoped.

### Common Mistakes
- Working directly on main for risky changes.
- Force-pushing shared branches.


In [ ]:
from pathlib import Path
import subprocess

root = Path.cwd()
if not (root / ".git").exists() and (root.parent / ".git").exists():
    root = root.parent

status = subprocess.run(["git", "status", "--short"], cwd=root, capture_output=True, text=True, check=False)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=root, capture_output=True, text=True, check=False)

print("Branch:", branch.stdout.strip() or "unavailable")
print("Status output:\n", status.stdout.strip() or "(clean or unavailable)")


## Section 3: GitHub Fundamentals

### Definition
GitHub is remote hosting platform for Git repositories.

### Theory
Deployment platforms integrate with GitHub webhooks and build from selected branch.

### Motivation
No GitHub repo, no Streamlit Cloud auto-deploy workflow.

### Real-World Example
Push commit to `main` -> Streamlit Cloud detects change -> rebuild starts automatically.

### Best Practices
- Add clear README.
- Use releases/tags for milestone versions.
- Keep sensitive files out of repository.

### Common Mistakes
- Committing `.streamlit/secrets.toml`.
- Publicly exposing keys in notebooks.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "README.md").exists() and (root.parent / "README.md").exists():
    root = root.parent

required_files = ["README.md", "app.py", "pyproject.toml", ".gitignore"]
report = {file: (root / file).exists() for file in required_files}
report
